In [1]:
import os
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("fitcheck")
    print("HF token loaded")
except Exception as error:
    print("No HF token:", error)
    print("Every model in the grid below is open, so this is not fatal.")

HF token loaded


In [2]:
import os

REPO = "/kaggle/working/fitcheck"

if not os.path.isdir(f"{REPO}/.git"):
    !git clone -q https://github.com/Anassbzdd/fitcheck.git /kaggle/working/fitcheck

%cd /kaggle/working/fitcheck
!git pull --ff-only || echo "git pull failed (dirty tree or diverged) -- using the code already on disk"
!git log --oneline -1

/kaggle/working/fitcheck
Already up to date.
686cf0d (HEAD -> main, origin/main, origin/HEAD) fix


In [3]:
!pip install -q -e .
!pip uninstall -q -y torchao


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fitcheck-llm (pyproject.toml) ... done


In [4]:
import argparse
import contextlib
import importlib
import io
import os
import sys

REPO = "/kaggle/working/fitcheck"
os.environ["PYTHONPATH"] = REPO
if f"{REPO}/scripts" not in sys.path:
    sys.path.insert(0, f"{REPO}/scripts")


def preflight() -> tuple[bool, str]:
    sweep = importlib.reload(importlib.import_module("calibration_sweep"))
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        ok = sweep.preflight(argparse.Namespace(quant="nf4")) is not None
    print(captured.getvalue().rstrip())
    return ok, captured.getvalue()


ok, report = preflight()

if not ok and "torchvision does not match torch" in report:
    print("\n--- dropping the mismatched torchvision/torchaudio, then retrying ---\n")
    !pip uninstall -q -y torchvision torchaudio
    ok, report = preflight()

if not ok and "bitsandbytes is NOT installed" in report:
    print("\n--- installing bitsandbytes (no -U, so torch is left alone) ---\n")
    !pip install -q bitsandbytes
    ok, report = preflight()

print()
print("STACK OK -- run the sweep." if ok else "STACK BROKEN -- fix the above first.")


PREFLIGHT FAILED -- `--quant nf4` needs bitsandbytes and it did not import.

  ModuleNotFoundError: No module named 'bitsandbytes'

bitsandbytes is NOT installed, and `--quant nf4` / `--quant int8` load the base
model through it. Kaggle and Colab ship torch, transformers and peft but not this
one, so an unquantized grid runs on a bare image and a quantized grid cannot.

    pip install bitsandbytes

Without `-U`. A plain install leaves the image's torch alone -- bitsandbytes
only requires a torch, and the image already has one that satisfies it -- while
`-U` upgrades torch itself and takes torchvision down with it (see the torchvision
note above). No kernel restart is needed: every row runs in a fresh subprocess.

Or take the unquantized grid instead, which needs nothing installed:

    --quant none   (and drop --tag nf4)

--- installing bitsandbytes (no -U, so torch is left alone) ---

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.4 MB/s eta 0:00:00:00:0100:01
Tesla T4 (

In [5]:
# Phase 3 -- the rows the C_overhead fit is still missing.
import argparse
import importlib
import json
import subprocess
import sys
import time
from pathlib import Path

REPO = "/kaggle/working/fitcheck"
if f"{REPO}/scripts" not in sys.path:
    sys.path.insert(0, f"{REPO}/scripts")
sweep = importlib.reload(importlib.import_module("calibration_sweep"))

OUT = Path("/kaggle/working/runs")
OUT.mkdir(parents=True, exist_ok=True)

TINY = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
QWEN = "Qwen/Qwen2.5-1.5B-Instruct"
SMOL = "HuggingFaceTB/SmolLM2-1.7B"

PLAN = []
# 3b -- seq-2048 anchors. Un-confounds sequence length from model size.
for model_id in (QWEN, SMOL):
    for kernel in ("eager", "flash"):
        for quant in ("nf4", "none"):
            PLAN.append((model_id, 1, 2048, kernel, quant))
# 3a -- the first batch ladder in the project. bs=16 left out: it lands at ~15,400 MiB
# on a 14,912 MiB card at the worst fragmentation this archive has measured.
for bs in (1, 2, 4, 8, 12):
    PLAN.append((TINY, bs, 1024, "flash", "nf4"))
for bs in (1, 2, 4, 8):
    PLAN.append((TINY, bs, 1024, "eager", "nf4"))

if sweep.preflight(argparse.Namespace(quant="nf4")) is None:
    raise SystemExit("preflight failed -- read the advice above, fix it, re-run")

done = skipped = failed = 0
consecutive = 0
clock = time.time()
for index, (model_id, bs, seq, kernel, quant) in enumerate(PLAN, 1):
    target = OUT / f"{sweep._slug(model_id, bs, seq, kernel, f'-{quant}')}.json"
    if target.exists():
        print(f"[{index}/{len(PLAN)}] {target.name}  (already done, skipping)")
        skipped += 1
        continue

    print(f"[{index}/{len(PLAN)}] {target.name} ...", flush=True)
    started = time.time()
    result = subprocess.run(
        sweep._row_args(
            model_id, bs, seq, kernel,
            argparse.Namespace(gpu="t4", quant=quant, precision="fp16", lora_r=32),
        ),
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        failed += 1
        oom = "out of memory" in (result.stderr or "").lower()
        print(f"    FAILED in {time.time() - started:.0f}s"
              f"{'  (OOM -- a real answer for a batch ladder, not an error)' if oom else ''}")
        for line in (result.stderr or "").strip().splitlines()[-4:]:
            print(f"      {line[:160]}")
        consecutive = 0 if oom else consecutive + 1
        if consecutive >= 3:
            print("\nSTOPPING: 3 rows failed in a row for reasons that are not memory.")
            print(sweep._diagnose(result.stderr))
            break
        continue

    payload = json.loads(result.stdout)
    target.write_text(json.dumps(payload, indent=1), encoding="utf-8")
    errors = payload.get("error_pct", {})
    done += 1
    consecutive = 0
    print(f"    ok in {time.time() - started:.0f}s  "
          f"tensors {errors.get('tensors', 0):+.1f}%  "
          f"activation {errors.get('activation', 0):+.1f}%  "
          f"process {errors.get('process', 0):+.1f}%")

print(f"\n{done} new rows, {skipped} already had, {failed} failed, "
      f"{time.time() - clock:.0f}s total")


Tesla T4 (sm_75, 14,912 MiB) | torch 2.10.0+cu128 | transformers 5.0.0 | peft 0.19.1 | bitsandbytes 0.50.2
[1/17] Qwen2-5-1-5B-Instruct-bs1-seq2048-eager-nf4.json ...
    ok in 46s  tensors -0.6%  activation -0.4%  process -1.0%
[2/17] Qwen2-5-1-5B-Instruct-bs1-seq2048-eager-none.json ...
    ok in 20s  tensors -0.1%  activation -0.2%  process +0.0%
[3/17] Qwen2-5-1-5B-Instruct-bs1-seq2048-flash-nf4.json ...
    ok in 22s  tensors -0.3%  activation -0.1%  process -1.3%
[4/17] Qwen2-5-1-5B-Instruct-bs1-seq2048-flash-none.json ...
    ok in 18s  tensors -0.0%  activation -0.0%  process +4.5%
[5/17] SmolLM2-1-7B-bs1-seq2048-eager-nf4.json ...
    ok in 49s  tensors +0.7%  activation +1.0%  process +2.0%
[6/17] SmolLM2-1-7B-bs1-seq2048-eager-none.json ...
    ok in 23s  tensors +1.1%  activation +2.7%  process -10.2%
[7/17] SmolLM2-1-7B-bs1-seq2048-flash-nf4.json ...
    ok in 22s  tensors +0.4%  activation +0.8%  process +2.1%
[8/17] SmolLM2-1-7B-bs1-seq2048-flash-none.json ...
    ok in 

In [6]:
import json
from pathlib import Path

from fitcheck.calibrate import fit_group, load_runs

RUNS = Path("/kaggle/working/runs")
RULE = "=" * 70

contexts = {}
for path in sorted(RUNS.glob("*.json")):
    c = json.loads(path.read_text(encoding="utf-8"))["measured"].get("cuda_context_mib")
    contexts[c] = contexts.get(c, 0) + 1
print("CUDA context across rows:", dict(sorted(contexts.items())))
if len(contexts) > 1 and max(contexts) - min(contexts) > 5:
    print("  WARNING: these rows come from sessions with different CUDA contexts.")
    print("  B fits that context, so one constant across them is not meaningful.")

groups, skipped = {}, 0
for path in sorted(RUNS.glob("*.json")):
    run = json.loads(path.read_text(encoding="utf-8"))["run"]
    if "-r2" in path.name or "-r3" in path.name:
        skipped += 1          # bit-identical repeats -- 3x weight in the least squares
        continue
    groups.setdefault((run["quantization"], run["kernel"]), []).append(path)

print(f"\n{sum(len(v) for v in groups.values())} rows, {skipped} repeats excluded")

for key in sorted(groups):
    runs = load_runs(groups[key])
    fit = fit_group(runs)
    p = fit.profile

    spread = []
    for i in range(len(runs)):
        try:
            spread.append(fit_group(runs[:i] + runs[i + 1:]).profile.fragmentation)
        except Exception:
            pass
    swing = ((max(spread) - min(spread)) / p.fragmentation * 100
             if spread and p.fragmentation else float("inf"))

    gates = {
        "B plausible (130-600)": 130 <= p.base_context_mib <= 600,
        "F positive": p.fragmentation > 0,
        "stable (LOO F swing <25%)": swing < 25,
        "worst process error <8%": fit.worst_abs_pct < 8,
    }
    print(f"\n{RULE}\n{key[0]} / {key[1]}   n={len(runs)}\n{RULE}")
    print(f"  B={p.base_context_mib:7.1f} MiB   F={p.fragmentation:+.4f}   "
          f"S={p.fragmentation_per_octave:+.4f}")
    print(f"  worst process error {fit.worst_abs_pct:.1f}%   LOO F swing {swing:.0f}%")
    print(f"  {'SHIPPABLE' if all(gates.values()) else 'NOT SHIPPABLE'}")
    for name, ok in gates.items():
        print(f"    {'OK ' if ok else 'X  '} {name}")


CUDA context across rows: {140.875: 17}

17 rows, 0 repeats excluded

nf4 / eager   n=6
  B=    0.0 MiB   F=+0.2957   S=+0.0000
  worst process error 13.7%   LOO F swing 92%
  NOT SHIPPABLE
    X   B plausible (130-600)
    OK  F positive
    X   stable (LOO F swing <25%)
    X   worst process error <8%

nf4 / flash   n=7
  B=   41.9 MiB   F=+0.2120   S=+0.0000
  worst process error 6.8%   LOO F swing 31%
  NOT SHIPPABLE
    X   B plausible (130-600)
    OK  F positive
    X   stable (LOO F swing <25%)
    OK  worst process error <8%

none / eager   n=2
  B= 1191.6 MiB   F=+0.0000   S=+0.0000
  worst process error 4.5%   LOO F swing inf%
  NOT SHIPPABLE
    X   B plausible (130-600)
    X   F positive
    X   stable (LOO F swing <25%)
    OK  worst process error <8%

none / flash   n=2
  B=  526.8 MiB   F=+0.0000   S=+0.0000
  worst process error 0.4%   LOO F swing inf%
  NOT SHIPPABLE
    OK  B plausible (130-600)
    X   F positive
    X   stable (LOO F swing <25%)
    OK  worst proc

In [7]:
!python -m fitcheck.calibrate /kaggle/working/runs/*.json --emit-python


OVERHEAD_DB: dict[tuple[str, str], OverheadProfile] = {
    ("t4", "eager"): OverheadProfile(
        gpu="Tesla T4",
        kernel="eager",
        base_context_mib=92.65,
        fragmentation=0.23737,
        fragmentation_per_octave=0.0,
        seq_len_min=1024,
        seq_len_max=2048,
        runs=8,
        worst_over_pct=11.6,
        worst_under_pct=-11.7,
        source="8 runs, 3 models, seq 1024-2048, fitcheck 0.3.0",
    ),
    ("t4", "flash"): OverheadProfile(
        gpu="Tesla T4",
        kernel="flash",
        base_context_mib=96.05,
        fragmentation=0.16614,
        fragmentation_per_octave=0.0,
        seq_len_min=1024,
        seq_len_max=2048,
        runs=9,
        worst_over_pct=10.1,
        worst_under_pct=-9.0,
        source="9 runs, 3 models, seq 1024-2048, fitcheck 0.3.0",
    ),
}


In [8]:
import time
from pathlib import Path

!cd /kaggle/working && rm -f runs.zip && zip -qr runs.zip runs
!ls -l --time-style=+%H:%M:%S /kaggle/working/runs.zip
print("rows inside:", len(list(Path("/kaggle/working/runs").glob("*.json"))))
print("built at", time.strftime("%H:%M:%S"),
      "-- if the Output panel shows an older time, refresh it")


-rw-r--r-- 1 root root 19114 12:54:39 /kaggle/working/runs.zip
rows inside: 17
built at 12:54:39 -- if the Output panel shows an older time, refresh it
